# Modeling

## Objetivo

Construir y comparar distintos enfoques de recomendación de productos.

Se evaluarán:

1. Modelo Baseline basado en popularidad.
2. Modelo Collaborative Filtering basado en interacciones usuario-producto.

Posteriormente se compararán los resultados para seleccionar el modelo más adecuado.

In [3]:
# Importar librerías necesarias

import pandas as pd
import numpy as np

In [4]:
# Cargar dataset procesado

model_data = pd.read_parquet(
    "../data/processed/model_data.parquet"
)

model_data.head()

,user_id,product_id,reordered,product_name,department,aisle
0,202279,33120,1,Organic Egg Whites,dairy eggs,eggs
1,202279,28985,1,Michigan Organic Kale,produce,fresh vegetables
2,202279,9327,0,Garlic Powder,pantry,spices seasonings
3,202279,45918,1,Coconut Butter,pantry,oils vinegars
4,202279,30035,0,Natural Sweetener,pantry,baking ingredients


In [5]:
# Construir ranking de productos más populares

popular_products = (
    model_data
    .groupby(
        ["product_id", "product_name"]
    )
    .size()
    .reset_index(name="purchase_count")
    .sort_values(
        by="purchase_count",
        ascending=False
    )
)

popular_products.head(10)

,product_id,product_name,purchase_count
17993,24852,Banana,449085
9475,13176,Bag of Organic Bananas,355348
15275,21137,Organic Strawberries,251953
15852,21903,Organic Baby Spinach,227037
34116,47209,Organic Hass Avocado,206151
34527,47766,Organic Avocado,167581
34426,47626,Large Lemon,145021
18986,26209,Limes,134698
20232,27966,Organic Raspberries,132434
12124,16797,Strawberries,131899


In [6]:
# Función para recomendar productos populares

def recommend_popular_products(n=10):

    return popular_products[
        ["product_id", "product_name", "purchase_count"]
    ].head(n)

In [7]:
recommend_popular_products(10)

,product_id,product_name,purchase_count
17993,24852,Banana,449085
9475,13176,Bag of Organic Bananas,355348
15275,21137,Organic Strawberries,251953
15852,21903,Organic Baby Spinach,227037
34116,47209,Organic Hass Avocado,206151
34527,47766,Organic Avocado,167581
34426,47626,Large Lemon,145021
18986,26209,Limes,134698
20232,27966,Organic Raspberries,132434
12124,16797,Strawberries,131899


# Modelo Baseline

## Descripción

Se implementó un sistema de recomendación basado en popularidad.

El modelo recomienda los productos con mayor cantidad de compras históricas dentro del dataset.

## Ventajas

- Implementación simple.
- Bajo costo computacional.
- No requiere historial del usuario.

## Limitaciones

- No existe personalización.
- Todos los usuarios reciben las mismas recomendaciones.
- No captura preferencias individuales.
- Favorece únicamente productos populares.

## Objetivo

Este modelo se utilizará como baseline para comparar el desempeño de modelos personalizados.

In [8]:
# Crear muestra reproducible para Collaborative Filtering

sample_users = (
    model_data["user_id"]
    .drop_duplicates()
    .sample(
        n=10000,
        random_state=42
    )
)

cf_data = model_data[
    model_data["user_id"].isin(sample_users)
].copy()

cf_data.shape

(2203808, 6)

In [9]:
# Crear variable de interacción

cf_data["interaction"] = 1

cf_data.head()

,user_id,product_id,reordered,product_name,department,aisle,interaction
237,129389,13176,1,Bag of Organic Bananas,produce,fresh fruits,1
238,129389,30442,1,Low Fat Vanilla Yogurt,dairy eggs,yogurt,1
239,129389,13646,1,Lemon Hummus,deli,fresh dips tapenades,1
240,129389,21019,0,Roasted Red Pepper Hummus,deli,fresh dips tapenades,1
241,129389,38226,0,Dark Chocolate Chips,snacks,candy chocolate,1


In [10]:
# Construir matriz usuario-producto

user_product_matrix = cf_data.pivot_table(
    index="user_id",
    columns="product_id",
    values="interaction",
    fill_value=0
)

user_product_matrix.shape

(10000, 32566)

In [11]:
# Calcular sparsity de la matriz

matrix_size = (
    user_product_matrix.shape[0]
    * user_product_matrix.shape[1]
)

filled_cells = (
    user_product_matrix
    .astype(bool)
    .sum()
    .sum()
)

sparsity = 1 - (
    filled_cells / matrix_size
)

print(f"Sparsity: {sparsity:.4%}")

Sparsity: 99.7365%


## Construcción de la matriz usuario-producto

Se construyó una matriz usuario-producto utilizando una muestra reproducible de 10.000 usuarios activos.

### Resultados

- Usuarios: 10.000
- Productos: 32.566
- Sparsity: 99,74%

La elevada dispersión observada es consistente con la naturaleza de los sistemas de recomendación reales, donde cada usuario interactúa únicamente con una pequeña fracción del catálogo disponible.

In [12]:
# Librerías para Collaborative Filtering

from sklearn.metrics.pairwise import cosine_similarity

In [13]:
# Calcular similitud entre usuarios

user_similarity = cosine_similarity(
    user_product_matrix
)

In [14]:
# Convertir matriz a DataFrame

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_product_matrix.index,
    columns=user_product_matrix.index
)

user_similarity_df.iloc[:5, :5]

user_id,21,41,63,80,109
user_id,,,,,
21,1.000000,0.058056,0.035447,0.000000,0.042216
41,0.058056,1.000000,0.124709,0.060684,0.020628
63,0.035447,0.124709,1.000000,0.059281,0.020152
80,0.000000,0.060684,0.059281,1.000000,0.019612
109,0.042216,0.020628,0.020152,0.019612,1.000000


In [15]:
sample_user = user_product_matrix.index[0]

sample_user

np.int64(21)

In [16]:
# Buscar usuarios similares

user_similarity_df[
    sample_user
].sort_values(
    ascending=False
).head(10)

user_id
21        1.000000
146495    0.157809
188609    0.157329
9502      0.153133
62211     0.150703
163069    0.150435
105002    0.145536
63586     0.139434
74355     0.138683
7239      0.137987
Name: 21, dtype: float64

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_product_matrix
)

user_similarity.shape

(10000, 10000)

In [18]:
# Función para recomendar productos usando usuarios similares

def recommend_user_based(user_id, n_recommendations=10, n_similar_users=10):
    
    # Obtener usuarios más similares, excluyendo al mismo usuario
    similar_users = (
        user_similarity_df[user_id]
        .sort_values(ascending=False)
        .iloc[1:n_similar_users + 1]
        .index
    )
    
    # Productos ya comprados por el usuario
    user_products = set(
        user_product_matrix.loc[user_id][
            user_product_matrix.loc[user_id] > 0
        ].index
    )
    
    # Productos comprados por usuarios similares
    similar_users_products = user_product_matrix.loc[similar_users]
    
    product_scores = (
        similar_users_products
        .sum(axis=0)
        .sort_values(ascending=False)
    )
    
    # Excluir productos ya comprados por el usuario
    recommendations = product_scores[
        ~product_scores.index.isin(user_products)
    ].head(n_recommendations)
    
    # Agregar nombres de productos
    recommendations = (
        recommendations
        .reset_index()
        .rename(columns={0: "score"})
        .merge(
            model_data[["product_id", "product_name"]].drop_duplicates(),
            on="product_id",
            how="left"
        )
    )
    
    return recommendations[["product_id", "product_name", "score"]]

In [19]:
recommend_user_based(sample_user, n_recommendations=10)

,product_id,product_name,score
0,36865,Non Fat Raspberry Yogurt,6.0
1,45066,Honeycrisp Apple,6.0
2,39408,Gala Apples,5.0
3,4957,Total 2% Lowfat Greek Strained Yogurt With Blu...,4.0
4,16797,Strawberries,4.0
5,17224,Oats & Honey Gluten Free Granola,4.0
6,21709,Sparkling Lemon Water,4.0
7,8490,Strawberry Rhubarb Yogurt,3.0
8,7350,Natural Lime Flavor Sparkling Mineral Water,3.0
9,5322,Gluten Free Dark Chocolate Chunk Chewy with a ...,3.0


# Modelo Collaborative Filtering User-Based

## Descripción

Se implementó un modelo de filtrado colaborativo basado en similitud entre usuarios.

El modelo identifica usuarios con patrones de compra similares y recomienda productos consumidos por esos usuarios, excluyendo los productos que el usuario objetivo ya compró previamente.

## Ventajas

- Genera recomendaciones personalizadas.
- Utiliza patrones reales de consumo.
- Puede descubrir productos relevantes más allá del ranking general de popularidad.

## Limitaciones

- Requiere historial suficiente del usuario.
- Puede verse afectado por la alta sparsity de la matriz usuario-producto.
- Presenta desafíos para usuarios nuevos sin historial de compra.

## Persistencia de artefactos del modelo

Con el objetivo de reutilizar los componentes entrenados fuera del entorno de análisis, se almacenan los principales artefactos generados durante el modelado.

Estos archivos serán utilizados posteriormente por la API de recomendaciones y por futuras aplicaciones de visualización, evitando recalcular la matriz usuario-producto y la matriz de similitud cada vez que se inicia el sistema.

### Artefactos almacenados

- user_product_matrix.parquet
- user_similarity_df.parquet

### Beneficios

- Reduce tiempos de carga.
- Evita cálculos repetitivos.
- Facilita el despliegue del modelo.
- Separa el entrenamiento de la etapa de inferencia.

In [20]:
user_product_matrix.shape

(10000, 32566)

In [22]:
user_similarity_df.shape

(10000, 10000)

In [21]:
# Guardar artefactos del modelo para uso en API

user_product_matrix.to_parquet(
    "../models/user_product_matrix.parquet"
)

user_similarity_df.to_parquet(
    "../models/user_similarity_df.parquet"
)

In [23]:
import os

print(os.path.exists("../models/user_product_matrix.parquet"))
print(os.path.exists("../models/user_similarity_df.parquet"))

True
True


### Resultado

Los artefactos fueron almacenados correctamente y quedan disponibles para ser consumidos por la API desarrollada en FastAPI.

Esta estrategia permite desacoplar el entrenamiento del modelo de la generación de recomendaciones en producción.